# OONCA: agente de apoio ao operador on-chain

Protótipo de análise de uma operação hipotética antes da assinatura. Usa `first_order_df.csv` e `wallet_metadata.csv`. O score é uma regra demonstrativa, não uma probabilidade de fraude. `isError` significa erro histórico. O agente não assina nem envia transações.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from app.agent.graph import analyze
from app.ingestion.schema import ProposedOperation
from app.models.risk import RiskPolicy
from app.visualization.risk import plot_assessment


Edite origem, destino e valor antes da execução. Com `use_llm=False`, a análise não chama o OpenRouter; mude para `True` para gerar uma explicação em linguagem natural. `as_of` opcional restringe o histórico a transações anteriores à data informada.


In [ ]:
operation = ProposedOperation.from_values(
    wallet_from='0xc2d7cf95645d33006175b78989035c7c9061d3f9',
    wallet_to='0x002f0c8119c16d310342d869ca8bf6ace34d9c39',
    amount='0.5',
    operation='transfer',
    asset='ETH',
)
result = analyze(operation, policy=RiskPolicy(), use_llm=False)
print(result['explanation'])
print('Histórico:', result['profile']['n_rows'], 'linhas; ', result['profile']['n_unique_tx_hashes'], 'hashes únicos')
print('Metadados:', result['profile']['wallet_metadata'])
plot_assessment(result)


## Limitações atuais

A amostra contém transações de 2016–2019 e pode repetir `TxHash` em várias linhas. O agente reporta contagem de linhas e hashes separadamente. As carteiras sem metadados não recebem um rótulo inventado. ENS e tipo de carteira foram consultados no presente, então podem não representar o estado na data das transações. Sem dados de token, calldata, gas, preço, rótulo de fraude ou operador, a decisão é uma triagem preliminar. Um `APPROVE` indica somente que nenhuma regra deste protótipo pediu revisão; não autoriza uma transação real.
